# Notebook 1 — Risk Marker Analysis & Funnel Profiling

**Purpose**: Establish whether hypertension risk markers carry meaningful signal about customer outcomes across 39 decision engines.

## What this notebook covers

- **Identity funnel** — how much customer-side context is needed before decisions become deterministic
- **Part A** — Association between each risk marker and the final decision (Cramér's V ranking)
- **Part B** — Which demographic variable resolves ambiguous marker profiles (diastolic BP is the answer)
- **Part C** — Whether rule changes over time explain residual ambiguity
- **Part D** — Severity analysis within non-standard decisions using loading bands
- **Part E** — Placing every customer on a 0–1 standard↔decline axis using logistic regression

## Data
Data is not included in this repository. See `data/README.md` for schema.

## Key findings
- Risk markers outperform all demographics on association strength (top marker Cramér's V ≈ 0.71 vs best demographic ≈ 0.07)
- Markers + demographics resolve ~80% of decisions deterministically; adding engine identity reaches ~98%
- Loading severity (em_load) does **not** track marker-based risk — Spearman ρ ≈ 0.05 pooled across engines
- ~40% of (profile × engine) cells are internally inconsistent, setting an upper bound on marker-only classifier performance

# POC13 — Hypertension Risk Profiling: Markers, Demographics & BP

**Goal:** Show how the 9 hypertension risk markers we collect contribute to underwriting decisions, and how demographics, time, and engine choice each layer additional explanatory power on top.

## Structure

- **Dataset overview** — what's in the data, decision breakdown, identity-level purity funnel
- **Part A — Pure (markers + engine) combos:** which markers actually drive a decision when the combination already determines the outcome
- **Part B — Adding demographics** to the identity: how much extra variability does this resolve?
- **Part C — Adding date** to the identity: does month-of-application explain the residual? Time-trend plots per engine
- **Part D — em_load severity** within non-standard accepts: bands, marker associations, demographic associations
- **Part E — The standard↔decline axis:** marker spectrum across the full severity range, plus a logistic regression that places every customer on a 0–1 decline-likeness axis. Tests whether em_load tracks marker-based risk


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, kruskal
from sklearn.cluster import KMeans
import warnings
warnings.filterwarnings('ignore')

RISK_MARKERS = [
    '(Advised to) Stopping treatment',
    'Compliant with treatment',
    'History of/current high blood pressure',
    'Hospitalised previously due to blood pressure',
    'Hypertension related conditions/symptoms',
    'Knows blood pressure',
    'Need/taking blood pressure medication',
    'Requires follow-up',
    'Waiting referral or investigation',
]
MARKER_SHORT = ['StopTx','Compliant','HBP_Hx','Hosp','HBP_Related','KnowsBP','Meds','Followup','WaitRef']

DECISION_ORDER = ['standard','non-standard','refer/evidence_required','postpone','decline']
DECISION_PAL   = {
    'standard':                '#2ecc71',
    'non-standard':            '#f39c12',
    'refer/evidence_required': '#e74c3c',
    'postpone':                '#3498db',
    'decline':                 '#8e44ad',
}

def cramers_v(chi2, n, r, c):
    return np.sqrt(chi2 / (n * (min(r, c) - 1))) if min(r, c) > 1 else 0.0

def chi2_cramers(col_a, col_b):
    ct = pd.crosstab(col_a, col_b)
    if ct.shape[0] < 2 or ct.shape[1] < 2:
        return None, None, None
    n = int(ct.values.sum())
    chi2, p, _, _ = chi2_contingency(ct)
    return chi2, p, cramers_v(chi2, n, ct.shape[0], ct.shape[1])

def readable_profile(s):
    m = {'T':'T','F':'F','NAsk':'N'}
    return ' '.join(m.get(p,p) for p in s.split('|'))

## Load & preprocess

We load the raw data and drop rows where every risk marker is missing — these rows carry no information for the hypertension analysis. Everything that follows works on this filtered set.


In [ ]:
DATA_PATH = "../data/FULL_LIFE_TABLE.csv"
raw = pd.read_csv(DATA_PATH)

# Drop rows with no risk markers
df = raw.dropna(subset=RISK_MARKERS, how='all').reset_index(drop=True)
print(f"Rows after dropping missing risk markers: {len(df):,}")
print(f"\nDecision distribution:")
print(df['decision'].value_counts())

print(f"Raw rows       : {len(raw):,}")
print(f"After BP filter: {len(df):,}  ({len(df)/len(raw)*100:.1f}% retained)")

# Decision category
present_decisions = [d for d in DECISION_ORDER if d in df['decision'].unique()]
df['decision'] = pd.Categorical(df['decision'], categories=present_decisions, ordered=True)

# year_month_cont
if 'year_month_cont' not in df.columns:
    df['created_date']    = pd.to_datetime(df['created_date'], errors='coerce')
    df['year_month_cont'] = df['created_date'].dt.year + (df['created_date'].dt.month - 1) / 12

# Risk marker profile string
df['profile'] = df[RISK_MARKERS].apply(lambda r: '|'.join(r.astype(str)), axis=1)

print(f"\nUnique profiles       : {df['profile'].nunique():,}")
print(f"Unique enquiry engines: {df['enquiry_engine'].nunique():,}")


## Dataset overview

Each row in this data is **one engine's evaluation of one application**. A single application can be sent to multiple engines simultaneously (a comparison/quote scenario), producing multiple rows with the same `application_key` but different `enquiry_engine` and potentially different decisions.

### The identity funnel

We start with a key question: **how much customer-side context do we need before the engine's decision becomes deterministic?**

We build progressively richer "identities" and at each layer count:
- **Pure** combinations — always produce one decision
- **Impure** combinations — produce more than one decision (something else is differentiating)

The shrinkage at each layer tells us how much that layer of context resolves decision variability.

We run this twice: once **without** `enquiry_engine` in the key (asking "is the customer-side signal alone enough?") and once **with** engine in the key (asking "given a specific engine, is the customer-side signal enough?").


In [ ]:
# ============================================================================
# Identity sense-check — funnel view (corrected)
# ============================================================================

LAYERS = [
    ('PURE',     'risk markers + engine',
        RISK_MARKERS_FULL + [ENGINE_COL]),
    ('PURE-ish', 'risk markers + engine + demographics',
        RISK_MARKERS_FULL + [ENGINE_COL] + DEMOGRAPHICS),
    ('IDENTITY', 'risk markers + engine + demographics + date',
        RISK_MARKERS_FULL + [ENGINE_COL] + DEMOGRAPHICS + [DATE_COL]),
]


def summarise(label, description, key_cols, df):
    g = df.groupby(key_cols, dropna=False, observed=True)
    n_per_group = g['decision'].nunique()
    impure_idx = n_per_group[n_per_group >  1].index
    pure_idx   = n_per_group[n_per_group == 1].index

    is_impure_row = df.set_index(key_cols).index.isin(impure_idx)
    pure_rows   = df[~is_impure_row]
    impure_rows = df[ is_impure_row]

    pure_cust   = set(pure_rows['customer_key'])
    impure_cust = set(impure_rows['customer_key'])
    pure_app    = set(pure_rows['application_key'])
    impure_app  = set(impure_rows['application_key'])

    return {
        'label'       : label,
        'description' : description,
        'combos'      : g.ngroups,
        'pure_combos' : len(pure_idx),
        'impure_combos': len(impure_idx),
        'pure_rows'   : len(pure_rows),
        'impure_rows' : len(impure_rows),

        # Customers: split into 3 mutually exclusive buckets
        'cust_only_pure'   : len(pure_cust - impure_cust),
        'cust_only_impure' : len(impure_cust - pure_cust),
        'cust_in_both'     : len(pure_cust & impure_cust),

        # Same for applications
        'app_only_pure'   : len(pure_app - impure_app),
        'app_only_impure' : len(impure_app - pure_app),
        'app_in_both'     : len(pure_app & impure_app),
    }


# Run all layers
results = [summarise(lbl, desc, key, df) for lbl, desc, key in LAYERS]

# True totals (constant across layers)
TOTAL_ROWS = len(df)
TOTAL_CUST = df['customer_key'].nunique()
TOTAL_APP  = df['application_key'].nunique()

print("="*72)
print("DATASET TOTALS (constant across all layers)")
print("="*72)
print(f"  Rows         : {TOTAL_ROWS:>12,}")
print(f"  Customers    : {TOTAL_CUST:>12,}")
print(f"  Applications : {TOTAL_APP:>12,}")

# --- Header table: combination counts ---
print("\n" + "="*72)
print("THE FUNNEL — adding context shrinks the impure space")
print("="*72)
print("\n  Pure   = combo always produces ONE decision (deterministic)")
print("  Impure = combo produces MULTIPLE decisions (variable)\n")

header = (f"{'Layer':<10} {'Definition':<46} "
          f"{'Combos':>10} {'Impure':>9} {'% Pure':>7}")
print(header)
print("-" * len(header))
for r in results:
    pct_pure = r['pure_combos'] / r['combos'] * 100
    print(f"{r['label']:<10} {r['description']:<46} "
          f"{r['combos']:>10,} {r['impure_combos']:>9,} {pct_pure:>6.1f}%")

# --- Detailed counts per layer (no double-counting) ---
print("\n" + "="*72)
print("PER-LAYER DETAIL")
print("="*72)
for r in results:
    print(f"\n[{r['label']}] — {r['description']}")
    print(f"  Combinations:  {r['combos']:>10,}  "
          f"({r['pure_combos']:,} pure | {r['impure_combos']:,} impure)")
    print(f"  Rows:          {TOTAL_ROWS:>10,}  "
          f"({r['pure_rows']:,} in pure combos | {r['impure_rows']:,} in impure combos)")
    print(f"  Customers:     {TOTAL_CUST:>10,}  (no double-counting):")
    print(f"      only in pure combos     : {r['cust_only_pure']:>10,}")
    print(f"      only in impure combos   : {r['cust_only_impure']:>10,}")
    print(f"      in both (straddlers)    : {r['cust_in_both']:>10,}")
    sum_check = r['cust_only_pure'] + r['cust_only_impure'] + r['cust_in_both']
    print(f"      ── sum                  : {sum_check:>10,}  "
          f"{'✓ matches total' if sum_check == TOTAL_CUST else '⚠ mismatch'}")
    print(f"  Applications:  {TOTAL_APP:>10,}  (no double-counting):")
    print(f"      only in pure combos     : {r['app_only_pure']:>10,}")
    print(f"      only in impure combos   : {r['app_only_impure']:>10,}")
    print(f"      in both                 : {r['app_in_both']:>10,}")

# --- The "monotonic growth" check ---
print("\n" + "="*72)
print("MONOTONIC CHECK — customers fully resolved at each layer")
print("="*72)
print("\n  Customers with ALL their rows in pure combos at this layer:")
print("  (this number should grow as we add more granularity)\n")
for r in results:
    n = r['cust_only_pure']
    pct = n / TOTAL_CUST * 100
    print(f"    {r['label']:<10}: {n:>10,}  ({pct:5.1f}% of all customers)")

### What the funnel tells us

Three findings:

1. **Markers alone resolve nearly half of decisions across all engines.** ~45% of marker combinations always produce one decision regardless of engine, demographics, or timing. The customer-side signal is real and largely engine-invariant.

2. **Adding demographics is the single biggest information layer.** Markers + demographics moves us from ~45% pure to ~80% pure — a ~35 percentage point jump. Demographics — and especially BP — are the workhorse explanatory variable.

3. **Adding date adds a small further bump.** Markers + demographics + date reaches ~86%. Time-of-application accounts for ~5pp more — engines occasionally update rules over time, but it's a secondary effect.

The remaining ~14% (without engine) or ~1.7% (with engine) is genuine residual variability — engine non-determinism or factors not in our data. Parts A through E investigate where this signal lives and what it means for risk.


In [ ]:
# ── (profile × engine) purity baseline ────────────────────────────────────────
engine_profile = (
    df.groupby(['profile','enquiry_engine'], observed=True)
    .agg(
        n_rows            =('decision','count'),
        dominant_decision =('decision', lambda x: x.value_counts().index[0]),
        dominant_pct      =('decision', lambda x: x.value_counts().iloc[0]/len(x)*100),
        n_unique_dec      =('decision','nunique'),
    )
    .reset_index()
)
pure_ep    = engine_profile[engine_profile['dominant_pct'] == 100]
nonpure_ep = engine_profile[engine_profile['dominant_pct'] <  100]

print(f"(profile × engine) combos : {len(engine_profile):,}")
print(f"  Pure   (100% one dec)   : {len(pure_ep):,}  ({len(pure_ep)/len(engine_profile)*100:.1f}%)")
print(f"  Non-pure (mixed dec)    : {len(nonpure_ep):,}  ({len(nonpure_ep)/len(engine_profile)*100:.1f}%)")

# Raw rows for each subset
df_pure = df.merge(pure_ep[['profile','enquiry_engine']], on=['profile','enquiry_engine'], how='inner').copy()
df_np   = df.merge(nonpure_ep[['profile','enquiry_engine']], on=['profile','enquiry_engine'], how='inner').copy()
print(f"\nRows in pure subset   : {len(df_pure):,}")
print(f"Rows in non-pure subset: {len(df_np):,}")

---
## Part A — Pure combinations: risk markers, demographics & risk tagging

These are `(profile × engine)` combos where 100% of rows share one decision.
They represent the deterministic, unambiguous cases.

In [ ]:
# A1 — Decision breakdown in pure combos
print("Pure combo decision breakdown:")
print(pure_ep['dominant_decision'].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: combo count by decision
pure_ep['dominant_decision'].value_counts().reindex(present_decisions, fill_value=0).plot(
    kind='bar', ax=axes[0], color=[DECISION_PAL[d] for d in present_decisions])
axes[0].set_title('Pure combos by decision', fontweight='bold')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)

# Right: row count by decision
pure_ep.groupby('dominant_decision')['n_rows'].sum().reindex(present_decisions, fill_value=0).plot(
    kind='bar', ax=axes[1], color=[DECISION_PAL[d] for d in present_decisions])
axes[1].set_title('Rows in pure combos by decision', fontweight='bold')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=20)

plt.suptitle('Part A — Pure (profile × engine) combos', fontsize=11)
plt.tight_layout()
plt.show()

### A2 — Which markers drive decisions?

We measure the strength of association between each marker (and demographic) and the decision outcome.

**Cramér's V** for categorical features (markers, gender, smoker): a number from 0–1 measuring relationship strength. >0.3 is strong, >0.5 is very strong.
**η² (eta-squared)** for continuous features (age, BMI, systolic, diastolic): the proportion of variance in that feature explained by the decision. >0.06 medium, >0.14 large.

These are effect sizes — they tell us *how strong* the relationship is. With 200k+ rows of pure combos, statistical significance is trivial; effect size is what actually matters.

Below: the ranking, plus a residuals heatmap showing which (marker value, decision) combinations are over- or under-represented vs random chance.


In [ ]:
# A2 — Association with decision: Cramér's V (categorical) + η² (continuous)
from scipy.stats import kruskal

cat_feats  = RISK_MARKERS + ['gender', 'smoker']
cat_labels = MARKER_SHORT + ['Gender', 'Smoker']
cat_types  = (['marker']*9) + (['demo']*2)

rows = []
for col, label, ftype in zip(cat_feats, cat_labels, cat_types):
    chi2, p, V = chi2_cramers(df[col], df['decision'])
    if V is not None:
        rows.append({'feature': label, 'type': ftype, 'stat': "V", 'value': round(V,4), 'p': p})

n = len(df)
cont_feats  = ['age_at_application', 'bmi', 'sys_pressure', 'dias_pressure']
cont_labels_c = ['Age', 'BMI', 'Systolic', 'Diastolic']
for col, label in zip(cont_feats, cont_labels_c):
    groups = [df.loc[df['decision']==d, col].dropna().values for d in present_decisions]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) >= 2:
        H, p = kruskal(*groups)
        eta2 = H / (n - 1)
        rows.append({'feature': label, 'type': 'cont', 'stat': "η²", 'value': round(eta2, 4), 'p': p})

feat_df = pd.DataFrame(rows).sort_values('value', ascending=False)
print("Features ranked by association with decision:\n")
print(feat_df.to_string(index=False))

colour_map = {l: "#91e186" for l in MARKER_SHORT}
colour_map.update({'Gender': "#7ebfeb", 'Smoker': '#3498db'})
colour_map.update({'Age': '#9b59b6', 'BMI': '#9b59b6', 'Systolic': "#ffadd5", 'Diastolic': '#e74c3c'})

fig, ax = plt.subplots(figsize=(8, len(feat_df)*0.45+2))
ax.barh(feat_df['feature'], feat_df['value'],
        color=[colour_map.get(f, '#95a5a6') for f in feat_df['feature']])
ax.axvline(0.1, color='grey', ls='--', lw=0.8)
ax.axvline(0.3, color='grey', ls=':',  lw=0.8)
ax.set_xlabel("Effect size  (Cramér's V for categorical | η² for continuous)")
ax.set_title("Association with decision — all\nGreen=markers | Blue=demographics | Purple=age/BMI | Red=BP", fontsize=10)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# A2b — Standardized residuals: which (marker=value, decision) combos are over/under-represented?
residual_rows = []
for col, short in zip(RISK_MARKERS, MARKER_SHORT):
    ct = pd.crosstab(df_pure[col], df_pure['decision'])
    ct = ct.reindex(columns=present_decisions, fill_value=0)
    _, _, _, expected = chi2_contingency(ct)
    pct_dev = (ct.values - expected) / np.maximum(expected, 1e-9) * 100
    z_vals  = (ct.values - expected) / np.sqrt(np.maximum(expected, 1e-9))

    for i, tag in enumerate(ct.index):
        for j, dec in enumerate(ct.columns):
            residual_rows.append({'marker_tag': f'{short}={tag}', 'marker': short,
                                   'tag': str(tag), 'decision': dec,
                                   'pct_dev': pct_dev[i,j], 'z': z_vals[i,j]})

resid_df   = pd.DataFrame(residual_rows)
resid_pct  = resid_df.pivot(index='marker_tag', columns='decision', values='pct_dev').reindex(columns=present_decisions)
resid_z    = resid_df.pivot(index='marker_tag', columns='decision', values='z').reindex(columns=present_decisions)

tag_rank = {'T':0,'F':1,'NAsk':2}
mkr_rank = {s:i for i,s in enumerate(MARKER_SHORT)}
resid_pct['_m'] = resid_pct.index.str.split('=').str[0].map(mkr_rank)
resid_pct['_t'] = resid_pct.index.str.split('=').str[1].map(tag_rank).fillna(99)
order = resid_pct.sort_values(['_m','_t']).index
resid_pct = resid_pct.loc[order].drop(columns=['_m','_t'])
resid_z   = resid_z.loc[order]

annot = resid_pct.round(0).astype(int).astype(str) + '%'

fig, ax = plt.subplots(figsize=(len(present_decisions)*2.2+3, len(resid_pct)*0.52+2.5))
sns.heatmap(resid_pct, annot=annot, fmt='', cmap=sns.diverging_palette(220,20,as_cmap=True),
            center=0, vmin=-150, vmax=150, linewidths=0.5, ax=ax,
            cbar_kws={'label': '% deviation from expected'})
for i, rl in enumerate(resid_pct.index):
    for j, cl in enumerate(resid_pct.columns):
        if pd.notna(resid_z.loc[rl,cl]) and abs(resid_z.loc[rl,cl]) >= 2:
            ax.text(j+0.5, i+0.88, '★', ha='center', va='bottom', fontsize=7, color='black', alpha=0.5)
ax.set_title('% deviation from expected — pure combos\n★=significant | Orange=over | Blue=under', fontsize=10)
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# A2c — Profile matrix: % True per marker × decision (pure combos)
pct_T = {}
for col, short in zip(RISK_MARKERS, MARKER_SHORT):
    pct_T[short] = {dec: (df_pure.loc[df_pure['decision']==dec, col]=='T').mean()*100
                    for dec in present_decisions}
profile_hm = pd.DataFrame(pct_T).reindex(present_decisions)

fig, ax = plt.subplots(figsize=(len(MARKER_SHORT)*1.4+2, len(present_decisions)+2.5))
sns.heatmap(profile_hm, annot=True, fmt='.0f', cmap='YlOrRd', vmin=0, vmax=100,
            linewidths=0.5, ax=ax, cbar_kws={'label': '% marker = True'})
ax.set_title('Decision profile: % True per marker × decision (pure combos only)', fontsize=10)
ax.tick_params(axis='x', rotation=40, labelsize=9)
plt.tight_layout()
plt.show()

# Standard vs every other decision: Δ% True
pct_T_df = pd.DataFrame(pct_T).reindex(present_decisions)
for dec in [d for d in present_decisions if d != 'standard']:
    delta = (pct_T_df.loc[dec] - pct_T_df.loc['standard']).sort_values(key=abs, ascending=False)
    print(f"\nstandard → {dec}  (Δ% True, largest shift first):")
    for marker, diff in delta.items():
        sign = '+' if diff >= 0 else '-'
        bar  = '█' * int(abs(diff)/2)
        print(f"  {marker:<14} {sign}{abs(diff):5.1f}%  {bar}")

### A3 — Demographics within pure combos

The marker ranking above showed that within pure combos, markers vastly out-rank demographics. The demographics aren't doing much work in this subset because the markers have already settled the decision.

The violin plots below confirm this — the distributions of age, BMI, and BP across decisions are similar within pure combos. Demographics come back into their own in **Part B**, where they resolve the cases markers can't fully decide.


In [ ]:
# A3 — Continuous demographics by decision (pure combos) — violin plots
cont_cols   = ['age_at_application', 'bmi', 'sys_num', 'dias_num']
cont_labels = ['Age', 'BMI', 'Systolic BP', 'Diastolic BP']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for ax, col, label in zip(axes, cont_cols, cont_labels):
    groups = [df_pure.loc[df_pure['decision']==d, col].dropna().values for d in present_decisions]
    stat, p = kruskal(*[g for g in groups if len(g) > 0])
    sns.violinplot(data=df_pure, x='decision', y=col, order=present_decisions,
                   palette=DECISION_PAL, inner='quartile', ax=ax)
    ax.set_title(f'{label} by decision (pure combos)\nKruskal-Wallis p={p:.2e}')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.show()


In [ ]:
cat_cols   = ['gender', 'smoker']
cat_labels = ['Gender', 'Smoker']

decisions_to_plot = [d for d in present_decisions if d != 'refer/evidence_required']
df_pure_nonrefer  = df_pure[df_pure['decision'].isin(decisions_to_plot)]

PASTEL = [
    "#2FA7F7",
    "#E2BEF1",
    '#FFF0A0',
    "#FF9AB3",
    "#8CB273",
    '#FFAD8F',
]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes = axes.flatten()

for i, (col, label) in enumerate(zip(cat_cols, cat_labels)):
    ct = pd.crosstab(df_pure_nonrefer['decision'], df_pure_nonrefer[col], normalize='index') * 100
    ct = ct.reindex(index=decisions_to_plot, fill_value=0)
    n_cats = len(ct.columns)
    ct.plot(kind='bar', ax=axes[i], color=PASTEL[:n_cats], width=0.7, legend=False)
    axes[i].set_title(label, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('% within decision')
    axes[i].tick_params(axis='x', rotation=20, labelsize=9)
    handles = [plt.Rectangle((0,0),1,1, color=PASTEL[j]) for j in range(n_cats)]
    axes[i].legend(handles, ct.columns.tolist(), fontsize=7, loc='upper right',
                   title=label, title_fontsize=7)

fig.suptitle('Categorical demographics per decision — pure combos (refer excluded)', fontsize=11)
plt.tight_layout()
plt.show()


### Note on risk tiering

We initially attempted to cluster customers into low/medium/high risk tiers using KMeans on markers + demographics. After several iterations (severity-weighted KMeans, score-based tiers, 2-population segmentation), we found that **the data has two distinct severity axes** — haemodynamic (BP, age, BMI) and procedural (waiting on referrals, non-compliance, treatment status). These don't align cleanly into a single tier ordering.

The more rigorous answer to the underlying question — *do markers contain real risk information?* — is in **Part E**, where we use logistic regression to place every customer on a single standard↔decline axis. We've removed the failed tiering attempts to keep the notebook focused on the analyses that landed.


---
## Part B — Non-pure combos: does a demographic variable explain the different decisions?

Same `(profile × engine)` but different demographics → different decision.

**B1:** Which single demographic variable resolves the most non-pure combos?
**B2:** Show decision distribution split by that differentiator.
**B3:** Chi-square across all demographic variables within non-pure rows.

In [ ]:
# B1 — Which single demographic variable resolves most non-pure combos?
demo_vars   = ['gender', 'smoker', 'age_at_application', 'bmi', 'sys_num', 'dias_num']
demo_labels = ['Gender', 'Smoker', 'Age', 'BMI', 'Systolic', 'Diastolic']

resolution = []
for col, label in zip(demo_vars, demo_labels):
    df_np['_test_key'] = (df_np['profile'] + '||' + df_np['enquiry_engine'] + '||'
                          + df_np[col].astype(str))
    grp = (df_np.groupby('_test_key')['decision']
               .apply(lambda x: x.value_counts().iloc[0]/len(x)*100))
    n_pure   = (grp == 100).sum()
    pct_pure = n_pure / len(grp) * 100
    resolution.append({'variable': label, 'col': col,
                        'n_combos_pure': n_pure, 'pct_combos_pure': round(pct_pure, 1)})

df_np.drop(columns=['_test_key'], inplace=True)
res_df = pd.DataFrame(resolution).sort_values('pct_combos_pure', ascending=False)

print("% of non-pure combos that become pure when ONE variable is added:\n")
print(res_df[['variable','pct_combos_pure','n_combos_pure']].to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(res_df['variable'], res_df['pct_combos_pure'], color='#3498db')
ax.set_xlabel('% of non-pure combos resolved')
ax.set_title('B1 — Which single variable resolves most non-pure combos?', fontsize=10)
ax.bar_label(ax.containers[0], fmt='%.1f%%', padding=3, fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


In [ ]:
# B2 — Decision distribution by demographic variable (non-pure rows)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, label in zip(axes, ['gender', 'smoker'], ['Gender', 'Smoker']):
    ct = pd.crosstab(df_np[col], df_np['decision'], normalize='index')*100
    ct = ct.reindex(columns=present_decisions, fill_value=0)
    ct.plot(kind='bar', stacked=True, ax=ax,
            color=[DECISION_PAL[d] for d in ct.columns], legend=False)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('% of rows')
    ax.tick_params(axis='x', rotation=0)
handles = [plt.Rectangle((0,0),1,1, color=DECISION_PAL[d]) for d in present_decisions]
fig.legend(handles, present_decisions, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5,-0.05))
fig.suptitle('B2a — Decision by categorical demographics (non-pure rows)', fontsize=11)
plt.tight_layout()
plt.show()

cont_cols   = ['age_at_application', 'bmi', 'sys_num', 'dias_num']
cont_labels = ['Age', 'BMI', 'Systolic BP', 'Diastolic BP']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for ax, col, label in zip(axes, cont_cols, cont_labels):
    sns.violinplot(data=df_np, x='decision', y=col, order=present_decisions,
                   palette=DECISION_PAL, inner='quartile', ax=ax)
    ax.set_title(f'{label} by decision (non-pure rows)', fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
fig.suptitle('B2b — Decision by continuous demographics (non-pure rows)', fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# B3 — Chi-square + Cramér's V for each demographic vs decision (non-pure rows only)
rows_b = []
for col, label in zip(demo_vars, demo_labels):
    chi2, p, V = chi2_cramers(df_np[col], df_np['decision'])
    if V is not None:
        rows_b.append({'feature': label, 'cramers_V': round(V,4), 'chi2': round(chi2,1), 'p_raw': p})

b_df = pd.DataFrame(rows_b).sort_values('cramers_V', ascending=False)
print("Cramér's V — demographics vs decision (non-pure rows only):\n")
b_df.style.bar(subset=['cramers_V'], color='#f39c12', vmin=0, vmax=0.3)

---
## Part C — Non-pure combos: is time (created_date) the explanation?

After adding all demographics + BP, some combos may still have mixed decisions.
Here we look at whether the decision changed over time for the same profile × engine —
a signal that the engine updated its UW philosophy.

Even if no clear trend is visible, that is also informative.

In [ ]:
# C1 — Build full extended key: profile + all demographics + exact BP values
df_np['extended_key'] = (
    df_np['profile']                         + '||' +
    df_np['bmi'].astype(str)                 + '|' +
    df_np['age_at_application'].astype(str)  + '|' +
    df_np['gender'].astype(str)              + '|' +
    df_np['smoker'].astype(str)              + '|' +
    df_np['sys_num'].fillna(-1).astype(str)  + '|' +
    df_np['dias_num'].fillna(-1).astype(str)
)

ext_ep = (
    df_np.groupby(['extended_key','enquiry_engine'], observed=True)
    .agg(n_rows       =('decision','count'),
         dominant_pct =('decision', lambda x: x.value_counts().iloc[0]/len(x)*100),
         n_unique_dec =('decision','nunique'))
    .reset_index()
)
ext_pure    = ext_ep[ext_ep['dominant_pct'] == 100]
ext_nonpure = ext_ep[ext_ep['dominant_pct'] <  100]

pct_resolved = len(ext_pure)/len(ext_ep)*100 if len(ext_ep) else 0
print(f"Non-pure (profile × engine) combos              : {len(nonpure_ep):,}")
print(f"Extended key combos (after adding exact demo+BP) : {len(ext_ep):,}")
print(f"  Resolved (now pure)                           : {len(ext_pure):,}  ({pct_resolved:.1f}%)")
print(f"  Still non-pure                                : {len(ext_nonpure):,}  ({100-pct_resolved:.1f}%)")
print(f"  Rows in still-non-pure                        : {ext_nonpure['n_rows'].sum():,}")


In [ ]:
# C2 — Time trends: top 5 engines by unique (customer, application) count
top5_engines = (
    df.groupby('enquiry_engine')
    .apply(lambda x: x[['customer_key','application_key']].drop_duplicates().shape[0])
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)
print("Top 5 engines by unique customer-application count:")
for e in top5_engines:
    n = df[df['enquiry_engine']==e][['customer_key','application_key']].drop_duplicates().shape[0]
    print(f"  {e}: {n:,}")

fig, axes = plt.subplots(len(top5_engines), 1,
                          figsize=(14, len(top5_engines)*3.2), sharex=False)
if len(top5_engines) == 1:
    axes = [axes]

for ax, engine in zip(axes, top5_engines):
    eng_np = df_np[df_np['enquiry_engine'] == engine].copy()
    if len(eng_np) == 0:
        ax.set_title(f'{engine} — no non-pure rows'); continue

    eng_np['ym'] = eng_np['year_month_cont'].round(2)
    trend = (eng_np.groupby(['ym','decision'])
             .size().rename('n').reset_index())
    totals = trend.groupby('ym')['n'].transform('sum')
    trend['pct'] = trend['n'] / totals * 100

    for dec in present_decisions:
        d = trend[trend['decision']==dec]
        if len(d) > 0:
            ax.plot(d['ym'], d['pct'], color=DECISION_PAL[dec],
                    marker='o', markersize=3, lw=1.5, label=dec)

    ax.set_title(f'{engine} — decision % over time (non-pure combos only)', fontsize=9, fontweight='bold')
    ax.set_xlabel('year_month_cont')
    ax.set_ylabel('% of rows')
    ax.legend(fontsize=7, loc='upper right', ncol=2)

plt.suptitle('C2 — Do decisions shift over time for the same profile × engine?\n'
             '(No clear trend is also informative)', fontsize=10, y=1.01)
plt.tight_layout()
plt.show()

---
## Part D — em_load severity analysis (non-standard decisions only)

Within the non-standard group, em_load reflects how severe the loading is.
We find natural breakpoints, then check which markers and demographics associate with higher loading.

In [ ]:
# D1 — em_load distribution in non-standard
ns_df = df[df['decision']=='non-standard'].copy()
ns_nonzero = ns_df[ns_df['em_load'] > 0]

print(f"Non-standard rows         : {len(ns_df):,}")
print(f"With em_load > 0          : {len(ns_nonzero):,}  ({len(ns_nonzero)/len(ns_df)*100:.1f}%)")
print(f"\nem_load distribution (em_load > 0):")
print(ns_nonzero['em_load'].value_counts().sort_index().to_string())

fig, ax = plt.subplots(figsize=(10, 4))
em_counts = ns_nonzero['em_load'].value_counts().sort_index()
ax.bar(em_counts.index.astype(str), em_counts.values, color='#f39c12', width=0.6)
ax.set_xlabel('em_load value')
ax.set_ylabel('Count')
ax.set_title('em_load distribution — non-standard decisions (em_load > 0)', fontsize=10)
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# D2 — Natural breakpoints via KMeans on em_load
X_em = ns_nonzero['em_load'].values.reshape(-1,1)
km_em = KMeans(n_clusters=4, random_state=42, n_init=10)
ns_nonzero = ns_nonzero.copy()
ns_nonzero['em_cluster'] = km_em.fit_predict(X_em)

cluster_means_em = ns_nonzero.groupby('em_cluster')['em_load'].mean().sort_values()
em_label_map = {c: l for c, l in zip(cluster_means_em.index,
                                      ['Light','Moderate','Heavy','Very Heavy'])}
em_colours   = {'Light':'#f9e79f','Moderate':'#f39c12','Heavy':'#e67e22','Very Heavy':'#c0392b'}
ns_nonzero['em_band'] = ns_nonzero['em_cluster'].map(em_label_map)

print("Natural break em_load bands:\n")
for band in ['Light','Moderate','Heavy','Very Heavy']:
    sub = ns_nonzero[ns_nonzero['em_band']==band]['em_load']
    print(f"  {band:<12}: em_load {sub.min():.0f} – {sub.max():.0f}  (n={len(sub):,})")

fig, ax = plt.subplots(figsize=(10,4))
for band in ['Light','Moderate','Heavy','Very Heavy']:
    sub = ns_nonzero[ns_nonzero['em_band']==band]['em_load'].value_counts().sort_index()
    ax.bar(sub.index.astype(str), sub.values, color=em_colours[band], label=band, width=0.6)
ax.set_xlabel('em_load value')
ax.set_ylabel('Count')
ax.set_title('em_load coloured by natural-break band', fontsize=10)
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

# Also add em_band to ns_df for subsequent analysis
ns_df = ns_df.merge(ns_nonzero[['application_key','enquiry_engine','em_band']],
                    on=['application_key','enquiry_engine'], how='left')
ns_df['em_band'] = ns_df['em_band'].fillna('Zero loading')

In [ ]:
# D3 — Which risk markers associate with higher em_load band?
em_bands_order = ['Zero loading','Light','Moderate','Heavy','Very Heavy']
ns_df['em_band'] = pd.Categorical(ns_df['em_band'], categories=em_bands_order, ordered=True)

rows_d = []
for col, short in zip(RISK_MARKERS, MARKER_SHORT):
    chi2, p, V = chi2_cramers(ns_df[col], ns_df['em_band'])
    if V is not None:
        rows_d.append({'marker': short, 'cramers_V': round(V,4), 'chi2': round(chi2,1)})

d_df = pd.DataFrame(rows_d).sort_values('cramers_V', ascending=False)
print("Risk markers vs em_load band (non-standard only):\n")
print(d_df.to_string(index=False))

fig, axes = plt.subplots(3,3, figsize=(18,13))
axes = axes.flatten()
em_pal = dict(zip(em_bands_order, ['#95a5a6','#f9e79f','#f39c12','#e67e22','#c0392b']))

for i, (col, short) in enumerate(zip(RISK_MARKERS, MARKER_SHORT)):
    ct = pd.crosstab(ns_df[col], ns_df['em_band'], normalize='index')*100
    ct = ct.reindex(columns=em_bands_order, fill_value=0)
    ct.plot(kind='bar', stacked=True, ax=axes[i],
            color=[em_pal[b] for b in ct.columns], legend=False)
    axes[i].set_title(short, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('% within tag')
    axes[i].tick_params(axis='x', rotation=0)

handles = [plt.Rectangle((0,0),1,1,color=em_pal[b]) for b in em_bands_order]
fig.legend(handles, em_bands_order, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5,-0.02))
fig.suptitle('D3 — em_load band by risk marker tag (non-standard only)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# D4 — Demographics vs em_load band (non-standard only)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, col, label in zip(axes, ['gender', 'smoker'], ['Gender', 'Smoker']):
    ct = pd.crosstab(ns_df[col], ns_df['em_band'], normalize='index')*100
    ct = ct.reindex(columns=em_bands_order, fill_value=0)
    ct.plot(kind='bar', stacked=True, ax=ax,
            color=[em_pal[b] for b in ct.columns], legend=False)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('% within group')
    ax.tick_params(axis='x', rotation=0)
handles = [plt.Rectangle((0,0),1,1, color=em_pal[b]) for b in em_bands_order]
fig.legend(handles, em_bands_order, loc='lower center', ncol=5, fontsize=9, bbox_to_anchor=(0.5,-0.05))
fig.suptitle('D4a — em_load band by categorical demographics (non-standard only)', fontsize=11)
plt.tight_layout()
plt.show()

cont_cols   = ['age_at_application', 'bmi', 'sys_num', 'dias_num']
cont_labels = ['Age', 'BMI', 'Systolic BP', 'Diastolic BP']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
for ax, col, label in zip(axes, cont_cols, cont_labels):
    sns.violinplot(data=ns_df, x='em_band', y=col, order=em_bands_order,
                   palette=em_pal, inner='quartile', ax=ax)
    ax.set_title(f'{label} by em_load band (non-standard only)', fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
fig.suptitle('D4b — em_load band by continuous demographics (non-standard only)', fontsize=11)
plt.tight_layout()
plt.show()

print("\nKruskal-Wallis: continuous demographics vs em_load (non-standard, em_load > 0):")
ns_nz = ns_df[ns_df['em_load'] > 0]
for col, label in zip(cont_cols, cont_labels):
    if col in ns_nz.columns:
        groups = [ns_nz.loc[ns_nz['em_band']==b, col].dropna().values
                  for b in em_bands_order if b != 'Zero loading']
        groups = [g for g in groups if len(g) > 1]
        if len(groups) >= 2:
            stat, p = kruskal(*groups)
            print(f"  {label:<14}: Kruskal-Wallis p={p:.2e}  {'★ significant' if p<0.05 else ''}")


---
## Part E — The standard↔decline axis

The goal of this section: place every application on a single 0–1 axis where 0 = "looks like a standard accept" and 1 = "looks like a decline." Then ask: **does em_load track position on this axis within non-standard accepts?**

If yes → loadings reflect underlying severity (a +75 customer is closer to decline than a +25 customer).
If no → loadings are driven by something else (engine pricing rules, specific clinical thresholds) and don't reflect a smooth severity gradient.

### E1 — Marker trajectory across the spectrum

First, descriptive: order applications by severity (standard → NS-Light → NS-Moderate → NS-Heavy → NS-Very Heavy → decline) and plot how each marker's `%T`, `%F`, and `%NAsk` evolves across this spectrum. Smooth monotonic lines = gradient. Flat lines across non-standard with sharp changes only at decline = no gradient.

### E2 — Logistic regression on standards vs declines

Train a model on **standards vs declines only** (the two anchor populations) using the 9 markers + demographics. Held-out AUC measures how well the markers separate the two extremes. Then we apply the trained model to every customer to get `p_decline_like` — a balanced 0–1 score on the same axis.

We run this **with** and **without** `enquiry_engine` as a feature, to quantify how much engine-specific knowledge contributes on top of customer-side features.

Critical methodological note: we use `class_weight='balanced'` because declines are only 0.5% of the data. This makes the score meaningfully spread across 0–1, but it means **`p_decline_like` is not a real-world probability of being declined** — it's a relative position on a balanced axis. A score of 0.5 means "in a balanced world, equally likely to be either anchor," not "50% chance of decline."

### E3 — Within-engine ρ check

The headline test of the original question: **Spearman ρ between em_load and `p_decline_like`** within non-standard. We do this both pooled across engines and within each engine separately, to make sure any correlation we find isn't a Simpson's paradox artefact.


In [ ]:
# ============================================================================
# Part E1 — Marker spectrum: standard → non-standard bands → decline
# Question: as severity increases left-to-right, do markers shift smoothly?
# ============================================================================

# Build the 6-group spectrum.
# We want app-level data here, not row-level, so we use app_df-style aggregation.
# Re-derive non-standard em_band on app level.
spectrum_src = (df.groupby('application_key', observed=True)
                  .agg(decision  =('decision', lambda x: x.value_counts().index[0]),
                       em_load   =('em_load', 'max'),
                       sys_num   =('sys_num','mean'),
                       dias_num  =('dias_num','mean'),
                       age       =('age_at_application','mean'),
                       bmi       =('bmi','mean'),
                       **{c: (c,'first') for c in RISK_MARKERS})
                  .reset_index())

def assign_spectrum_group(row):
    d = row['decision']
    if d == 'standard': return 'standard'
    if d == 'decline':  return 'decline'
    if d == 'non-standard':
        em = row['em_load']
        if pd.isna(em) or em <= 0: return None
        if em <=  35: return 'NS-Light'
        if em <=  60: return 'NS-Moderate'
        if em <= 101: return 'NS-Heavy'
        return 'NS-Very Heavy'
    return None  # drop refer/postpone for this view — they're not on the severity axis

spectrum_src['spec_group'] = spectrum_src.apply(assign_spectrum_group, axis=1)
spec_df = spectrum_src.dropna(subset=['spec_group']).copy()

spec_order = ['standard','NS-Light','NS-Moderate','NS-Heavy','NS-Very Heavy','decline']
spec_df['spec_group'] = pd.Categorical(spec_df['spec_group'],
                                       categories=spec_order, ordered=True)

# Sample sizes per group
print("Spectrum group sizes:")
for g in spec_order:
    n = (spec_df['spec_group'] == g).sum()
    print(f"  {g:<14}: n={n:>8,}")

# Marker %T trajectory across the spectrum
fig, axes = plt.subplots(3, 3, figsize=(16, 11))
axes = axes.flatten()
spec_palette = ['#2ecc71','#f9e79f','#f39c12','#e67e22','#c0392b','#8e44ad']

for i, (col, short) in enumerate(zip(RISK_MARKERS, MARKER_SHORT)):
    pct_t = spec_df.groupby('spec_group', observed=True)[col].apply(
        lambda x: (x == 'T').mean() * 100).reindex(spec_order)
    pct_f = spec_df.groupby('spec_group', observed=True)[col].apply(
        lambda x: (x == 'F').mean() * 100).reindex(spec_order)
    pct_n = spec_df.groupby('spec_group', observed=True)[col].apply(
        lambda x: (x == 'NAsk').mean() * 100).reindex(spec_order)

    ax = axes[i]
    ax.plot(range(len(spec_order)), pct_t.values, marker='o', lw=2,
            color='#c0392b', label='% T')
    ax.plot(range(len(spec_order)), pct_f.values, marker='s', lw=2,
            color='#2ecc71', label='% F')
    ax.plot(range(len(spec_order)), pct_n.values, marker='^', lw=1.5,
            color='#7f8c8d', label='% NAsk', alpha=0.6)
    ax.set_xticks(range(len(spec_order)))
    ax.set_xticklabels(spec_order, rotation=30, fontsize=7)
    ax.set_title(short, fontweight='bold', fontsize=10)
    ax.set_ylim(0, 100)
    ax.set_ylabel('% of group')
    ax.grid(alpha=0.3)
    if i == 0:
        ax.legend(fontsize=7, loc='upper left')

fig.suptitle('Marker trajectory across severity spectrum\n'
             '(standard → non-standard bands → decline)', fontsize=11)
plt.tight_layout()
plt.show()

# Demographics across the spectrum
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col, label in zip(axes, ['sys_num','dias_num','age','bmi'],
                          ['Systolic','Diastolic','Age','BMI']):
    means = spec_df.groupby('spec_group', observed=True)[col].mean().reindex(spec_order)
    ax.plot(range(len(spec_order)), means.values, marker='o', lw=2, color='#2980b9')
    ax.set_xticks(range(len(spec_order)))
    ax.set_xticklabels(spec_order, rotation=30, fontsize=8)
    ax.set_title(f'Mean {label}', fontweight='bold', fontsize=10)
    ax.grid(alpha=0.3)
fig.suptitle('Demographics across severity spectrum', fontsize=11)
plt.tight_layout()
plt.show()

# Numerical table for the slide
print("\n%T per marker per spectrum group:\n")
pct_t_table = spec_df.groupby('spec_group', observed=True)[RISK_MARKERS].apply(
    lambda g: (g == 'T').mean() * 100
).reindex(spec_order).rename(columns=dict(zip(RISK_MARKERS, MARKER_SHORT)))
print(pct_t_table.round(1).to_string())

In [ ]:
# ============================================================================
# Part E2 — Quantitative spectrum placement (ROW-LEVEL)
#
# Each row is one engine's evaluation of one application.
# We do NOT dedup to application_key because different engines processing the
# same application produce genuinely separate decision events.
#
# Caveat: rows are not fully independent — a customer who applied to N engines
# contributes N correlated rows. Effective sample size is smaller than the row
# count. AUC may be slightly optimistic for this reason. The Spearman ρ on
# em_load is robust to this and remains the headline finding.
# ============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

# 1. Build feature matrix at row level (no dedup)
def build_features(src):
    rm = pd.get_dummies(src[RISK_MARKERS], drop_first=False)
    demo = pd.DataFrame({
        'age':  pd.to_numeric(src['age_at_application'], errors='coerce'),
        'bmi':  pd.to_numeric(src['bmi'],               errors='coerce'),
        'sys':  pd.to_numeric(src['sys_num'],           errors='coerce'),
        'dias': pd.to_numeric(src['dias_num'],          errors='coerce'),
    }, index=src.index)
    eng = pd.get_dummies(src['enquiry_engine'], prefix='engine', drop_first=False)
    return pd.concat([rm, demo, eng], axis=1).fillna(0)

X_full = build_features(df)

# 2. Train on all standards vs all declines
mask_std    = (df['decision'] == 'standard').values
mask_dec    = (df['decision'] == 'decline').values
mask_anchor = mask_std | mask_dec

X_anchor = X_full[mask_anchor]
y_anchor = mask_dec[mask_anchor].astype(int)

print(f"Anchor set (row-level): {mask_std.sum():,} standards + {mask_dec.sum():,} declines")
print(f"  Decline rate: {mask_dec.sum() / mask_anchor.sum() * 100:.2f}%")

# Hold out 20% to measure AUC
X_tr, X_te, y_tr, y_te = train_test_split(X_anchor, y_anchor, test_size=0.2,
                                          stratify=y_anchor, random_state=42)

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

model = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
model.fit(X_tr_s, y_tr)

auc = roc_auc_score(y_te, model.predict_proba(X_te_s)[:,1])
print(f"\nHeld-out AUC (standards vs declines): {auc:.3f}")
print("(>0.85 = strong; >0.95 = near-perfect; <0.7 = markers don't separate cleanly)")

# 3. Score every row in the dataset
X_full_s = scaler.transform(X_full)
df_scored = df.copy()
df_scored['p_decline_like'] = model.predict_proba(X_full_s)[:,1]

# 4. Score distribution by decision
print(f"\nScore distribution by decision (row-level):")
score_summary = df_scored.groupby('decision')['p_decline_like'].agg(
    ['mean','median','count']).round(3)
print(score_summary.to_string())

std_scores = df_scored.loc[df_scored['decision'] == 'standard',     'p_decline_like'].values
ns_scores  = df_scored.loc[df_scored['decision'] == 'non-standard', 'p_decline_like'].values
dec_scores = df_scored.loc[df_scored['decision'] == 'decline',      'p_decline_like'].values

# 5. em_load gradient — only non-standards with valid em_load
ns_with_em = df_scored[(df_scored['decision'] == 'non-standard') & 
                       (df_scored['em_load'] > 0)].copy()
ns_with_em['em_band'] = ns_with_em['em_load'].apply(
    lambda em: 'NS-Light'      if em <= 35
          else 'NS-Moderate'   if em <= 60
          else 'NS-Heavy'      if em <= 101
          else 'NS-Very Heavy')

band_order = ['NS-Light','NS-Moderate','NS-Heavy','NS-Very Heavy']
ns_with_em['em_band'] = pd.Categorical(ns_with_em['em_band'],
                                        categories=band_order, ordered=True)

print(f"\nMean p(decline-like) per em_load band ({len(ns_with_em):,} non-standard rows with em_load):")
band_summary = ns_with_em.groupby('em_band', observed=True)['p_decline_like'].agg(
    ['mean','median','count','std']).round(3)
print(band_summary.to_string())

# 6. Plots
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: density histograms
bins = np.linspace(0, 1, 41)
axes[0].hist(std_scores, bins=bins, alpha=0.55, color='#2ecc71',
             label=f'Standard (n={len(std_scores):,})', density=True)
axes[0].hist(ns_scores, bins=bins, alpha=0.55, color='#f39c12',
             label=f'Non-standard (n={len(ns_scores):,})', density=True)
axes[0].hist(dec_scores, bins=bins, alpha=0.55, color='#8e44ad',
             label=f'Decline (n={len(dec_scores):,})', density=True)
axes[0].set_xlabel('p(decline-like)  —  0 = looks standard | 1 = looks decline')
axes[0].set_ylabel('Density')
axes[0].set_title('Where does each group fall on the standard↔decline axis?\n'
                  '(row-level: each row = one engine evaluation)', fontsize=10)
axes[0].legend(fontsize=9)

# Right: em_load gradient
sns.boxplot(data=ns_with_em, x='em_band', y='p_decline_like',
            order=band_order, palette=['#f9e79f','#f39c12','#e67e22','#c0392b'],
            showfliers=False, ax=axes[1])
axes[1].axhline(std_scores.mean(), color='#2ecc71', ls='--', lw=1.5,
                label=f'Standard mean ({std_scores.mean():.2f})')
axes[1].axhline(dec_scores.mean(), color='#8e44ad', ls='--', lw=1.5,
                label=f'Decline mean ({dec_scores.mean():.2f})')
axes[1].set_xlabel('')
axes[1].set_ylabel('p(decline-like)')
axes[1].set_title('Does em_load track marker-based risk?\n'
                  '(rising = yes, flat = no)', fontsize=10)
axes[1].legend(fontsize=8, loc='upper left')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

# 7. Spearman correlation
rho, p_val = spearmanr(ns_with_em['em_load'], ns_with_em['p_decline_like'])
print(f"\nSpearman correlation between em_load and p(decline-like): "
      f"ρ = {rho:.3f}  (p = {p_val:.2e})")
print(f"Based on {len(ns_with_em):,} non-standard rows with valid em_load")
print("Interpretation:")
print("  ρ > 0.5    → em_load tracks marker-based risk strongly")
print("  ρ ≈ 0.2–0.5 → moderate alignment")
print("  ρ < 0.2    → loadings don't reflect underlying marker risk in a continuous way")

In [ ]:
def run_logreg(X, y_anchor_mask, y_dec_mask, label):
    X_anchor = X[y_anchor_mask]
    y_anchor = y_dec_mask[y_anchor_mask].astype(int)
    
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_anchor, y_anchor, test_size=0.2, stratify=y_anchor, random_state=42)
    
    sc = StandardScaler()
    Xtr = sc.fit_transform(X_tr); Xte = sc.transform(X_te)
    m = LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42)
    m.fit(Xtr, y_tr)
    
    auc = roc_auc_score(y_te, m.predict_proba(Xte)[:,1])
    
    # Score whole dataset
    scores = m.predict_proba(sc.transform(X))[:,1]
    return auc, scores, m, sc

mask_std = (df['decision'] == 'standard').values
mask_dec = (df['decision'] == 'decline').values
mask_anchor = mask_std | mask_dec

# Build both feature matrices
def build_features_no_engine(src):
    rm = pd.get_dummies(src[RISK_MARKERS], drop_first=False)
    demo = pd.DataFrame({
        'age':  pd.to_numeric(src['age_at_application'], errors='coerce'),
        'bmi':  pd.to_numeric(src['bmi'], errors='coerce'),
        'sys':  pd.to_numeric(src['sys_num'], errors='coerce'),
        'dias': pd.to_numeric(src['dias_num'], errors='coerce'),
    }, index=src.index)
    return pd.concat([rm, demo], axis=1).fillna(0)

def build_features_with_engine(src):
    base = build_features_no_engine(src)
    eng = pd.get_dummies(src['enquiry_engine'], prefix='engine', drop_first=False)
    return pd.concat([base, eng.set_index(src.index)], axis=1).fillna(0)

X_no_eng   = build_features_no_engine(df)
X_with_eng = build_features_with_engine(df)

print("Comparing model variants:")
print(f"  Anchor set: {mask_std.sum():,} standards + {mask_dec.sum():,} declines\n")

auc_no,   scores_no,   model_no,   sc_no   = run_logreg(X_no_eng,   mask_anchor, mask_dec, "No engine")
auc_with, scores_with, model_with, sc_with = run_logreg(X_with_eng, mask_anchor, mask_dec, "With engine")

print(f"  AUC without engine : {auc_no:.3f}")
print(f"  AUC with engine    : {auc_with:.3f}")
print(f"  Δ from engine      : +{auc_with - auc_no:.3f}")

# Compare em_load gradient under each
ns_idx = (df['decision'] == 'non-standard') & (df['em_load'] > 0)
em_load = df.loc[ns_idx, 'em_load'].values
rho_no,   _ = spearmanr(em_load, scores_no[ns_idx.values])
rho_with, _ = spearmanr(em_load, scores_with[ns_idx.values])
print(f"\n  Spearman ρ (em_load vs p_decline_like):")
print(f"    Without engine : {rho_no:.3f}")
print(f"    With engine    : {rho_with:.3f}")
print(f"  → both should be near zero; engine doesn't fix the em_load issue")

In [ ]:
# Sanity check: Spearman ρ within each individual engine, no pooling
import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# Use the no-engine model's scores (the cleaner attribution)
ns_rows = df[(df['decision'] == 'non-standard') & (df['em_load'] > 0)].copy()
ns_rows['p_decline_like'] = scores_no[(df['decision'] == 'non-standard') & (df['em_load'] > 0)]

# Top engines by volume, ρ within each
top_engines = ns_rows['enquiry_engine'].value_counts().head(8).index
print(f"Spearman ρ within each engine (using engine-AGNOSTIC model scores):\n")
print(f"{'Engine':<35} {'n':>10} {'ρ':>8}")
print("-" * 55)
for eng in top_engines:
    sub = ns_rows[ns_rows['enquiry_engine'] == eng]
    if len(sub) < 100: continue
    rho, _ = spearmanr(sub['em_load'], sub['p_decline_like'])
    print(f"{str(eng)[:34]:<35} {len(sub):>10,} {rho:>8.3f}")

---
## Synthesis

What we've established:

1. **The 9 markers carry strong customer-side risk information.** Without any engine knowledge, markers + demographics distinguish standards from declines with AUC ~0.92. With engine added, AUC bumps to ~0.95 — engine is incremental, not primary.

2. **Markers rank above every demographic for predicting decisions.** Even the weakest marker (Hosp at Cramér's V ≈ 0.17) is more than twice as strong as the best demographic (Smoker at ~0.07). The single strongest marker — WaitRef — has Cramér's V around 0.71, an exceptionally strong association.

3. **Demographics resolve most of what markers don't.** Going from "markers only" to "markers + demographics" jumps purity from ~45% to ~80%. BP (especially diastolic) is the workhorse — it alone resolves over 60% of impure (markers + engine) combinations.

4. **em_load does not track marker-based risk consistently.** Within non-standard, the Spearman correlation between em_load and `p_decline_like` is ~0.05 pooled across engines, and within most individual engines it's near zero or even negative. **Customers loaded +25 are statistically indistinguishable on markers from customers loaded +75.**

5. **The decline boundary and the loading scale are different mechanisms.** Markers + demographics identify *whether* a customer crosses the decline cliff (AUC 0.92). They do *not* identify *where on the loading scale* a non-standard customer sits — that's driven by engine-specific rules and clinical thresholds (likely BP severity bands, hospitalisation history) that don't align cleanly with the marker pattern that distinguishes standards from declines.

### Implication for downstream work

If you're modelling lifetime customer value or portfolio risk:
- **Use markers as a strong predictor of the decline boundary.** They generalise across engines.
- **Don't treat em_load as a normalised severity score across engines.** Different engines use different scales. If you need to compare loadings across engines, you'd need to normalise within each engine first.
- **For sub-segmenting non-standard customers**, em_load alone won't give you a meaningful risk gradient — you'd need to combine it with the marker-based score from Part E.
